Generate ShopFlow raw data as csv files in data/

In [1]:
import csv, random, sys
from datetime import datetime, timedelta
from pathlib import Path

In [2]:
random.seed(42)
OUT = Path("data")
OUT.mkdir(exist_ok=True)
START = datetime(2026, 1, 1)

In [ ]:
CITIES = [("Manila", "PH"), ("Cebu", "PH"), ("Naga", "PH"), ("Batangas", "PH"), ("London", "UK") , ("Sydney", "AU")]
FIRST = ["Ali", "Joe", "Mario", "Aurora", "Jon", "Pete", "Emmy", "Justin"]
LAST = ["Gabriel", "Jones", "Bond", "Markee", "St. Thomas"]
PRODUCTS = [("Wireless Mouse", "Electronics", 2452), 
            ("USB-C Hub", "Electronics", 100), 
            ("Desk Lamp", "Home", 120),
            ("Coffee Mug", "Home", 89),
            ("Notebook", "Stationery", 10),
            ("Gel Pens", "Stationery", 20),
            ("Backpack", "Bags", 150),
            ("Laptop Sleeve", "Bags", 1000),
            ]
STATUSES = ["placed", "Shipped", "shipped", "DELIVERED", "delivered", "returned"]
METHODS = ["card", "card", "wallet", "bank_transfer", "cash_on_delivery"]

In [ ]:
def write(name, header, rows):
    with open(OUT / name, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
        
    print(f"{name:<18} {len(rows):>5} rows")

In [ ]:
customers = []
for cid in range(1, 201):
    city, country = random.choice(CITIES)
    first, last = random.choice(FIRST), random.choice(LAST)
    signup = START + timedelta(days=random.randint(0, 120))
    email = f"{first}.{last}{cid}@example.com"
    if cid % 25 == 0:
        email = email.upper()
    customers.append([cid, first, last, email, city, country, signup.date(), "standard", signup])

write("customers.csv", ["customer_id", "first_name", "last_name", "email", "city", "country", "signup_date", "tier", "updated_at"], customers)

products = [[i + 1, n, c, p] for i, (n, c, p) in enumerate(PRODUCTS)]
write("products.csv", ["product_id", "product_name", "category", "price_cents"], products)

orders, items, payments = [], [], []
item_id = pay_id = 1
for oid in range(1, 1201):
    cust = random.choice(customers)
    placed = cust[8] + timedelta(days=random.randint(1, 180), minutes=random.randint(0, 1439))
    status = random.choice(STATUSES)
    orders.append([oid, cust[0], placed, status, placed])
    total = 0
    for _ in range(random.randint(1, 4)):
        p = random.choice(products)
        qty = random.randint(1, 3)
        items .append([item_id, oid, p[0], qty, p[3]])
        total += qty * p[3]
        item_id += 1
        payments.append([pay_id, oid, random.choice(METHODS), total, placed + timedelta(minutes=5)])
    pay_id += 1

orders.append(orders[10]) # duplicating
write("orders.csv", ["order_id", "customer_id", "order_date", "status", "updated_at"], orders)
write("order_items.csv", ["order_item_id", "order_id", "product_id", "quantity", "unit_price_cents"], items)
write("payments.csv", ["payment_id", "order_id", "method", "amount_cents", "created_at"], payments)

In [ ]:
# Run papermill
# > papermill generate_data.ipynb output_generate_data.ipynb